# Workshop: PySpark

### Monday, April 15, 2024

In today's exercises, we'll see a bit of the basics of PySpark.
In particular, we'll see how to run lots of parallel jobs in PySpark, and use this basic pattern to perform Monte Carlo estimation.

## Basics of PySpark on GCP

The following link includes two brief exercises for using PySpark on GCP.
The first runs in Python; the second runs in scala.

https://cloud.google.com/dataproc/docs/tutorials/monte-carlo-methods-with-hadoop-spark

Complete the first exercise (the one in Python), which will walk you through a simple pattern for parallelizing a randomized experiment and computing simple statistics on the results.

In [1]:
'''
ssudhir2@stochastic-cluster-m:~$ pyspark
Python 3.8.15 | packaged by conda-forge | (default, Nov 22 2022, 08:46:39) 
[GCC 10.4.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/15 15:22:33 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker
24/04/15 15:22:33 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster
24/04/15 15:22:33 INFO org.apache.spark.SparkEnv: Registering BlockManagerMasterHeartbeat
24/04/15 15:22:33 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /__ / .__/\_,_/_/ /_/\_\   version 3.1.3
      /_/

Using Python version 3.8.15 (default, Nov 22 2022 08:46:39)
Spark context Web UI available at http://stochastic-cluster-m.c.continual-voice-420414.internal:37031
Spark context available as 'sc' (master = yarn, app id = application_1713193815518_0001).
SparkSession available as 'spark'.
>>> import random
>>> import time
>>> from operator import add
>>> 
>>> def grow(seed):
...     random.seed(seed)
...     portfolio_value = INVESTMENT_INIT
...     for i in range(TERM):
...         growth = random.normalvariate(MKT_AVG_RETURN, MKT_STD_DEV)
...         portfolio_value += portfolio_value * growth + INVESTMENT_ANN
...     return portfolio_value
... 
>>> seeds = sc.parallelize([time.time() + i for i in range(10000)])
>>> results = seeds.map(grow)
>>> INVESTMENT_INIT = 100000  # starting amount
>>> INVESTMENT_ANN = 10000  # yearly new investment
>>> TERM = 30  # number of years
>>> MKT_AVG_RETURN = 0.11 # percentage
>>> MKT_STD_DEV = 0.18  # standard deviation
>>> 
>>> sum = results.reduce(add)
[Stage 0:>                                                          (0 + 2) /[Stage 0:============================================>              (3 + 1) /                                                                             >>> print( sum/10000 )
4273093.604663704
>>> 
>>> MKT_AVG_RETURN = 0.07
>>> 
>>> print (sc.parallelize([time.time() + i for i in range(10000)]) \
...         .map(grow).reduce(add)/10000.)
[Stage 1:=============================>                             (2 + 2) /                                                                             1700695.7024312923
>>> 
'''

<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
C:\Users\shriv\AppData\Local\Temp\ipykernel_3832\797425631.py:1: SyntaxWarning: invalid escape sequence '\ '
  '''


'\nssudhir2@stochastic-cluster-m:~$ pyspark\nPython 3.8.15 | packaged by conda-forge | (default, Nov 22 2022, 08:46:39) \n[GCC 10.4.0] on linux\nType "help", "copyright", "credits" or "license" for more information.\nSetting default log level to "WARN".\nTo adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).\n24/04/15 15:22:33 INFO org.apache.spark.SparkEnv: Registering MapOutputTracker\n24/04/15 15:22:33 INFO org.apache.spark.SparkEnv: Registering BlockManagerMaster\n24/04/15 15:22:33 INFO org.apache.spark.SparkEnv: Registering BlockManagerMasterHeartbeat\n24/04/15 15:22:33 INFO org.apache.spark.SparkEnv: Registering OutputCommitCoordinator\nWelcome to\n      ____              __\n     / __/__  ___ _____/ /__\n    _\\ \\/ _ \\/ _ `/ __/  \'_/\n   /__ / .__/\\_,_/_/ /_/\\_\\   version 3.1.3\n      /_/\n\nUsing Python version 3.8.15 (default, Nov 22 2022 08:46:39)\nSpark context Web UI available at http://stochastic-cluster-m.c.continual-voice-42041

## A Monte Carlo estimate of $\pi$

Consider the following experiment:

1. Select a point uniformly at random from the unit square $[-1,1] \times [-1,1]$.
2. If the point is within distance 1 of the origin, record a "success", otherwise record a "failure".

Observe that the probability of a "success" in the above experiment is exactly $\pi/4$. Thus, this experiment gives us a way to estimate the value of $\pi$: repeat the experiment $n$ times and let $X$ be the number of successes.
Then $\mathbb{E} X_n = n \pi / 4$, and $4 X_n / n$ is an estimate of $\pi$.
A similar approach to estimating $\pi$ was pioneered by [Buffon](https://en.wikipedia.org/wiki/Buffon%27s_needle_problem), a famous French mathematician.

Implement this experiment in a function, and modify the PySpark code from the previous exercise on GCP to get an estimate for $\pi$.

<b>Bonus exercise:</b> Use Chebyshev's inequality (or your favorite concentration inequality) to establish (approximately) how large $n$ has to be to obtain an estimate of $\pi$ that is accurate up to $4$ significant figures.
Run the above code with $n$ MC replicates and compare the result to the true value of $\pi = 3.14159265...$.